<a href="https://colab.research.google.com/github/roberaBelachew/clarity_cpc2_exploration.ipynb/blob/main/clarity_cpc2_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import shutil

disk = shutil.disk_usage("/content")

print(f"Free disk space: {disk.free / 1e9:.1f} GB")

Free disk space: 93.6 GB


In [4]:
# Downloading the dataset
!mkdir -p /content/clarity_cpc2

!wget -c \
  "https://zenodo.org/records/17045970/files/clarity_CPC2_data.v1_1.tgz?download=1" \
  -O /content/clarity_cpc2/clarity_CPC2_data.v1_1.tgz



--2026-10-27 09:20:12--  https://zenodo.org/records/17045970/files/clarity_CPC2_data.v1_1.tgz?download=1
Resolving zenodo.org (zenodo.org)... 188.185.40.132, 137.138.121.78, 188.185.51.60, ...
Connecting to zenodo.org (zenodo.org)|188.185.40.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12096370815 (11G) [application/octet-stream]
Saving to: ‘/content/clarity_cpc2/clarity_CPC2_data.v1_1.tgz’

larity_cpc2/clarity   0%[                    ]   4.38M  1016KB/s    eta 3h 12m ^C


In [6]:
# extracting the metadata files
from pathlib import Path
import tarfile

project_dir = Path("/content/clarity_cpc2")
archive_path = project_dir / "clarity_CPC2_data.v1_1.tgz"

with tarfile.open(archive_path, "r:gz") as archive:
    for member in archive:
        if "metadata" in Path(member.name).parts and member.isfile():
            archive.extract(member, path=project_dir, filter="data")

print("Metadata extraction complete.")


EOFError: Compressed file ended before the end-of-stream marker was reached

In [11]:
from google.colab import files

uploaded = files.upload()

Saving cpc2_metadata.zip to cpc2_metadata (1).zip


In [12]:
from pathlib import Path
import zipfile
import json
import pandas as pd

metadata_dir = Path("/content/cpc2_metadata")
metadata_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile("cpc2_metadata.zip") as archive:
    archive.extractall(metadata_dir)

with open(metadata_dir / "CEC2.train.1.json", encoding="utf-8") as file:
    responses = pd.DataFrame(json.load(file))

with open(metadata_dir / "listeners.json", encoding="utf-8") as file:
    listeners = json.load(file)

print("Ready!")
print("Response rows:", len(responses))
print("Listeners:", len(listeners))

Ready!
Response rows: 2779
Listeners: 49


In [13]:
listener_files = list(project_dir.rglob("listeners.json"))

if not listener_files:
    raise FileNotFoundError("listeners.json was not found after extraction.")

metadata_dir = listener_files[0].parent

print("Metadata folder:", metadata_dir)

for path in sorted(metadata_dir.glob("*.json")):
    print(path.name)

Metadata folder: /content/clarity_cpc2/clarity_CPC2_data/clarity_data/metadata
._listeners.json
CEC1.train.1.json
CEC1.train.2.json
CEC1.train.3.json
CEC2.train.1.json
CEC2.train.2.json
CEC2.train.3.json
listeners.json
scenes.CEC1.json
scenes.CEC2.json


In [14]:
import json
import pandas as pd

response_path = metadata_dir / "CEC2.train.1.json"

with open(response_path, encoding="utf-8") as file:
    records = json.load(file)

responses = pd.DataFrame(records)

print("Rows and columns:", responses.shape)
display(responses.head())

Rows and columns: (2779, 10)


,prompt,scene,n_words,hits,listener,system,response,volume,signal,correctness
0,i could feel its presence and now i can't,S08792,9,5,L0200,E008_hr,i could feel and now,96,S08792_L0200_E008_hr,55.555556
1,the engineer would go to sleep soon in a corner,S09161,10,2,L0200,E008_hr,their engineer was in at,96,S09161_L0200_E008_hr,20.000000
2,and to ask you to support dad,S09377,7,1,L0200,E008_hr,to cook over,96,S09377_L0200_E008_hr,14.285714
3,i was about to telephone the police,S08892,7,2,L0200,E008_hr,to telephone,96,S08892_L0200_E008_hr,28.571429
4,finally he put it in his mouth,S09432,7,0,L0200,E008_hr,,96,S09432_L0200_E008_hr,0.000000


In [15]:
import json

with open(metadata_dir / "listeners.json", encoding="utf-8") as file:
    listeners = json.load(file)

print("Number of listeners:", len(listeners))

Number of listeners: 49


In [16]:
first_response = responses.iloc[0]
listener_id = first_response["listener"]

print("Listener ID:", listener_id)
print(listeners[listener_id])

Listener ID: L0200
{'name': 'L0200', 'audiogram_cfs': [250, 500, 1000, 2000, 3000, 4000, 6000, 8000], 'audiogram_levels_l': [35, 30, 25, 50, 55, 65, 70, 65], 'audiogram_levels_r': [45, 45, 20, 50, 70, 65, 80, 75]}


CONNECTING THE AUDIOGRAM WITH THE PERSON


In [17]:
profile = listeners[listener_id]

audiogram = pd.DataFrame({
    "Frequency (Hz)": profile["audiogram_cfs"],
    "Left ear (dB HL)": profile["audiogram_levels_l"],
    "Right ear (dB HL)": profile["audiogram_levels_r"]
})

display(audiogram)

,Frequency (Hz),Left ear (dB HL),Right ear (dB HL)
0,250,35,45
1,500,30,45
2,1000,25,20
3,2000,50,50
4,3000,55,70
5,4000,65,65
6,6000,70,80
7,8000,65,75


In [19]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=audiogram)

https://docs.google.com/spreadsheets/d/1CNRe6zoT02kJXkOqiFzM6J3aDaOTVd_NbQaO-yL8bBY/edit#gid=0


In [9]:
# from pathlib import Path

# matches = list(Path("/content/clarity_cpc2").rglob("listeners.json"))
# print(matches)

[PosixPath('/content/clarity_cpc2/clarity_CPC2_data/clarity_data/metadata/listeners.json')]


In [10]:
# import shutil
# from google.colab import files

# metadata_dir = matches[0].parent

# backup_path = shutil.make_archive(
#     "/content/cpc2_metadata",
#     "zip",
#     root_dir=metadata_dir
# )

# files.download(backup_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>